# HomerGPT — essai TPU : vitesse, puis mini-run 8 cœurs avec reprise

In [ ]:
import subprocess, glob, shutil


def run(cmd: str) -> None:
    """Lance une commande en affichant sa sortie ; une erreur arrête le notebook (et la session)."""
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end='', flush=True)
    if p.wait():
        raise RuntimeError(f'échec ({p.returncode}) : {cmd}')


for name in ('train_tpu.py', 'prepare_data.py'):
    shutil.copy(glob.glob(f'/kaggle/input/**/{name}', recursive=True)[0], f'/kaggle/working/{name}')

In [ ]:
run("pip install -q torch==2.9.0 --index-url https://download.pytorch.org/whl/cpu && pip install -q 'torch_xla[tpu]==2.9.0'")
run("python -c \"import torch, torch_xla, torch_xla.runtime as xr; print(torch.__version__, torch_xla.__version__, xr.device_type(), xr.global_runtime_device_count()); assert xr.device_type() == 'TPU'\"")

In [ ]:
run("cd /kaggle/working && python train_tpu.py bench && cat bench.json")

In [ ]:
run("cd /kaggle/working && FAKE_PRETRAIN=1 MODEL=small OUT=/kaggle/working/essai1 SESSION=1 HOURS_PRETRAIN=0.08 TOTAL_PRETRAIN_HOURS=0.16 HOURS_FINETUNE=0.04 EVAL_EVERY_S=90 TOKENS_PER_STEP=131072 python train_tpu.py train")

In [ ]:
run("cd /kaggle/working && FAKE_PRETRAIN=1 MODEL=small OUT=/kaggle/working/essai2 SESSION=2 RESUME_DIR=/kaggle/working/essai1 HOURS_PRETRAIN=0.05 TOTAL_PRETRAIN_HOURS=0.16 HOURS_FINETUNE=0.03 EVAL_EVERY_S=90 TOKENS_PER_STEP=131072 python train_tpu.py train")

In [ ]:
run("rm -rf /kaggle/working/essai1/*.pt /kaggle/working/essai*/checkpoints/*.safetensors")

In [ ]:
run("rm -f /kaggle/working/train_tpu.py /kaggle/working/prepare_data.py; ls -laR /kaggle/working | head -40")